In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import glob
import tiktoken
from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr
import json
from langchain_community.document_loaders import TextLoader
from collections import Counter

In [ ]:
MODEL = "gpt-4.1-nano"
EMBEDING_MODEL = "text-embedding-3-small"
DB_NAME = "vector_db"
PERSIST_DIRECTORY = "../vector_db"
RETRIEVAL_K = 13
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 300
RELEVANCE_THRESHOLD = 1.5

In [ ]:
load_dotenv(override=True)
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

In [ ]:
knowledge_base_path = "../knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base.")

In [ ]:
full_knowledge_base = ""

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        full_knowledge_base += f.read()
        full_knowledge_base += "\n\n"

encoding = tiktoken.encoding_for_model(EMBEDING_MODEL)
tokens = encoding.encode(full_knowledge_base)
print(f"Total tokens for {EMBEDING_MODEL} knowledge base: {len(tokens)}")

In [ ]:
def clean_text(text):
    lines = text.split("\n")
    return "\n".join(" ".join(line.split()) for line in lines)

In [ ]:
folders = glob.glob("../knowledge-base/*")

documents = []
categories = ["All"]

for folder in folders:
    doc_category = os.path.basename(folder)
    categories.append(doc_category)

    loader = DirectoryLoader(
        path=folder,
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )

    folder_docs = loader.load()

    for folder_doc in folder_docs:
        folder_doc.metadata["category"] = doc_category
        folder_doc.page_content = clean_text(folder_doc.page_content)
        documents.append(folder_doc)

print(documents[1])

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP
)
chunks = text_splitter.split_documents(documents)

In [ ]:
embedding_model = OpenAIEmbeddings(model=EMBEDING_MODEL, api_key=OPENAI_KEY)

if os.path.exists(PERSIST_DIRECTORY) and os.listdir(PERSIST_DIRECTORY):
    print("Loading existing Chroma DB...")

    vectorstore = Chroma(
        collection_name=DB_NAME,
        embedding_function=embedding_model,
        persist_directory=PERSIST_DIRECTORY,
    )

else:
    print("Creating new Chroma DB...")

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        collection_name=DB_NAME,
        persist_directory=PERSIST_DIRECTORY,
    )

In [ ]:
def retrieve_docs(question: str, category: str | None = None, k: int = RETRIEVAL_K):

    search_kwargs = {"k": k}

    if category:
        search_kwargs["filter"] = {"category": category}

    return vectorstore.as_retriever(
        search_type="similarity", search_kwargs=search_kwargs
    ).invoke(question)

In [ ]:
llm = ChatOpenAI(model=MODEL, temperature=0)

In [ ]:
SYSTEM_PROMPT = """
You are a helpful and knowledgeable customer support assistant for TechNova.

Answer the user's question using ONLY the information provided in the Context section.

Rules:
1. If the answer is found in the context, provide a clear and concise response.
2. If the answer is partially available, answer only with the available information.
3. If the answer is not in the context, say exactly: "I don't have enough information to answer that question."
4. Do not make up information or use outside knowledge.
5. When multiple context sections are provided, use the most relevant information.
6. If the context contains conflicting information, mention the conflict instead of guessing.
7. For yes/no questions, always start with "Yes" or "No" when the context supports it. Do not start a negative answer with "Yes".
8. If the user asks about a product configuration (base, mid, pro, max), include both the configuration details and the price.
9. If the question asks for methods, options, items, features, or choices, list ALL matching items from the context — do not stop after the first one.
10. When answering about support tickets, base your answer primarily on the Resolution section of the ticket. Include the actions taken, outcome, and any repair fee, refund amount, replacement details, shipping method, or quoted cost mentioned there.
11. When answering about financing or payment plans, include the provider names and the minimum qualifying order amount.
12. When answering about shipping restrictions, include the exact status from the context — if it says "suspended", use that word explicitly.
13. When answering about sustainability or environmental goals, include the emissions scope classification (e.g. "Scope 3") if mentioned in the context.
14. If the context contains a numeric condition or threshold (e.g. "below 80%", "within 60 days", "over $50"), include it in your answer.
15. If the user's question contains a specific value and the context has a matching threshold, compare them explicitly and state whether the policy applies.
16. When the question asks whether something qualifies, is covered, or is eligible, explicitly state "covered", "not covered", "qualifies", or "does not qualify".
17. If the user asks about iPhones or iPhone compatibility, treat any mention of "iOS compatibility" in the context as a direct answer — iOS and iPhone refer to the same platform.
18. When answering release-note questions about added features or types, list all matching item names from the context. Do not omit items that appear in the source.
19. If the user sends a greeting, farewell, conversational message, or a question 
about the conversation itself (e.g. "Hi", "Hello", "Thanks", "do you know my name?", 
"what did I just say?"), respond naturally and briefly using the conversation history, 
and end your response with the tag [NO_SOURCES].
20. When the answer contains multiple steps or sequential action items (more than one 
distinct action to perform), format them as a numbered list (1. 2. 3.) and start 
the response with: "Please follow the instructions below:"
For single-sentence answers or factual responses, do NOT use this format.
21. If the user expresses emotional distress, personal crisis, or makes statements 
about self-harm (e.g. "i want to die", "i wanna die", "i hate my life"), respond 
with empathy and direct them to seek help. End your response with [NO_SOURCES].



Context:
{context}


"""

In [ ]:
def clean_source_path(path: str) -> str:
    return (
        path.replace("../knowledge-base/", "")
        .replace("knowledge-base/", "")
        .replace("../", "")
    )


def answer_question(question: str, history=None, category=None):

    if history is None:
        history = []

    # STEP 1: decide docs only
    if len(question.strip().split()) > 3:
        # Generic off-topic detection using similarity score
        docs_with_score = vectorstore.similarity_search_with_score(
            question, k=RETRIEVAL_K
        )
        best_score = docs_with_score[0][1] if docs_with_score else 999
        print(f"docs with score : {docs_with_score}")
        print(f"Best simialiry score : {best_score}")

        if best_score > RELEVANCE_THRESHOLD:
            return "I 'm a Technova support assistant.I can only help with Technova product and service related questions."

        if category is None or category == "" or category == "All":
            docs = retrieve_docs(question)
        else:
            docs = retrieve_docs(question, category)
    else:
        docs = retrieve_docs(question)

        print(f"docs : {docs}")

    # STEP 2: LLM call — runs for BOTH branches
    context = "\n\n".join(
        f"Source: {doc.metadata.get('source')}\n"
        f"Category: {doc.metadata.get('category')}\n"
        f"Content: {doc.page_content}"
        for doc in docs
    )

    system_prompt = SYSTEM_PROMPT.format(context=context)

    messages = [
        SystemMessage(content=system_prompt),
    ]

    print(history)

    for h in history:
        if h["role"] == "user":
            messages.append(HumanMessage(content=h["content"]))
        elif h["role"] == "assistant":
            messages.append(AIMessage(content=h["content"]))

    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)
    answer = response.content

    print(answer)
    if "[NO_SOURCES]" in answer:
        return answer.replace("[NO_SOURCES]", "").strip()

    if answer == "I don't have enough information to answer that question.":
        return answer

    # In answer_question:
    seen = dict.fromkeys(
        clean_source_path(doc.metadata.get("source", ""))
        for doc in docs
        if doc.metadata.get("source")
    )
    top_sources = list(seen.keys())[:3]
    sources_text = "\n".join(f"- {src}" for src in top_sources)

    return f"{answer}\n\nSources:\n{sources_text}"

In [ ]:
select_categories = gr.Dropdown(
    categories,
    label="Select Category",
)
gr.ChatInterface(
    fn=answer_question,
    title="TechNova RAG Assistant",
    additional_inputs=select_categories,
).launch()

In [ ]:
JUDGE_SYSTEM_PROMPT = """
You are an expert evaluator of RAG assistant answers.

You will be given:
- The user's question
- Required facts that MUST appear in the answer, but they may be paraphrased
- Expected source files
- Actual retrieved source files
- The assistant's generated answer

Your job is to score the answer from 0 to 5.

Scoring Rubric:
5 – Perfect: All required facts are present, clearly stated, and at least one expected source appears in the actual sources.
4 – Good: All required facts are present, but the answer is slightly vague OR source match is weak.
3 – Acceptable: Most required facts are present, but one important fact is missing.
2 – Poor: Only about half of the required facts are present.
1 – Very poor: Only one required fact is present or the answer is mostly incorrect.
0 – Completely wrong, unsupported, or says it does not know when the answer should exist.

Source Evaluation:
- If expected sources are provided, check whether at least one expected source appears in the actual sources.
- Do not penalize extra actual sources too heavily if the expected source is present.
- Penalize if none of the expected sources appear in the actual sources.
- Required facts are mandatory. If a required fact is missing from the assistant answer and is not clearly paraphrased, do not give a score of 5.
- For exact terms, numbers, prices, dates, product names, policy statuses, and legal/shipping statuses, require the exact meaning to appear. For example, if "suspended" is required, "due to local regulations" is not enough.
- If expected sources are empty, ignore source matching. For out-of-scope questions, a correct refusal should receive a high score and missing_facts should be empty.

Return ONLY a JSON object like this:
{
    "score": <integer 0-5>,
    "reason": "<brief explanation>",
    "missing_facts": ["fact1", "fact2"],
    "source_match": true
}
"""


def llm_judge(
    question: str,
    answer: str,
    must_mention: list[str],
    expected_sources,
    actual_sources,
):

    must_mention_text = ", ".join(must_mention)
    actual_sources_list = ", ".join(actual_sources)
    expected_sources_list = ", ".join(expected_sources)

    user_prompt = f"""
        Question: {question}

        Required facts (must be present, can be paraphrased):
        {must_mention_text}
        
        Expected Sources
        {expected_sources_list}
        
        Actual Sources
        { actual_sources_list }

        Assistant's answer:
        {answer}

        Score according to the rubric. Return JSON only.
    """

    messages = [
        SystemMessage(content=JUDGE_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ]

    response = llm.invoke(messages)

    # Parse JSON – handle possible markdown wrapping
    content = response.content.strip()
    if content.startswith("```json"):
        content = content[7:]
    if content.endswith("```"):
        content = content[:-3]

    try:
        result = json.loads(content)
    except:
        # Fallback
        result = {
            "score": 0,
            "reason": "Failed to parse judge output",
            "missing_facts": must_mention,
        }
    return result

In [ ]:
def evaluate_answer():
    with open("../evaluation/questions.json", "r", encoding="utf-8") as f:
        evaluation_data = json.load(f)
        questions = evaluation_data["questions"]

    results = []

    total_question = len(questions)
    retrieval_hits = 0

    out_of_scope_total = 0
    out_of_scope_refused = 0
    total_score = 0

    for q in questions:

        id = q["id"]
        question = q["question"]
        must_mention = q["must_mention"]
        expected_sources = q["expected_sources"]

        answer, actual_sources = answer_question(question, [])

        is_out_of_scope = len(expected_sources) == 0
        if is_out_of_scope:
            out_of_scope_total += 1
            if "I don't have enough information to answer that question" in answer:
                out_of_scope_refused += 1

        if any(src in actual_sources for src in expected_sources):
            retrieval_hits += 1

        # Judge it
        judgement = llm_judge(
            question, answer, must_mention, expected_sources, actual_sources
        )
        score = judgement["score"]
        total_score += score

        print(f"retrieveal_hits : {retrieval_hits}")
        results.append(
            {
                "id": id,
                "question": question,
                "predicted_answer": answer,
                "retrieved_sources": actual_sources,
                "score": score,
            }
        )

    refused_correctly = (
        out_of_scope_refused == out_of_scope_total if out_of_scope_total > 0 else False
    )
    average_score = round(total_score / total_question, 2)
    retrieval_precision = round(retrieval_hits / total_question, 3)
    output = {
        "summary": {
            "total_questions": total_question,
            "retrieval_precision": retrieval_precision,
            "answer_score_avg": average_score,
            "refused_correctly": refused_correctly,
        },
        "per_question": results,
    }

    with open("../evaluation/eval_results.json", "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2)

    return output


# evaluate_answer()

# Reports


In [ ]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter
import os, glob

# ============================================================
# CELL 2 — Knowledge Base Overview: Documents per Category
# ============================================================
folders = glob.glob("../knowledge-base/*")
category_counts = {}
for folder in folders:
    doc_category = os.path.basename(folder)
    files = glob.glob(os.path.join(folder, "**/*.md"), recursive=True)
    category_counts[doc_category] = len(files)

categories = list(category_counts.keys())
counts = list(category_counts.values())
colors = plt.cm.Set2(np.linspace(0, 1, len(categories)))

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(categories, counts, color=colors, edgecolor="white", linewidth=1.5)
ax.bar_label(bars, padding=4, fontsize=11, fontweight="bold")
ax.set_title("Documents per Category in Knowledge Base", fontsize=14, fontweight="bold")
ax.set_xlabel("Category")
ax.set_ylabel("Number of Documents")
ax.set_ylim(0, max(counts) + 2)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

# ============================================================
# CELL 3 — Token Count per Category
# ============================================================

encoding = tiktoken.encoding_for_model("text-embedding-3-small")

category_tokens = {}
for folder in folders:
    doc_category = os.path.basename(folder)
    total = 0
    for f in glob.glob(os.path.join(folder, "**/*.md"), recursive=True):
        with open(f, "r", encoding="utf-8") as fh:
            total += len(encoding.encode(fh.read()))
    category_tokens[doc_category] = total

fig, ax = plt.subplots(figsize=(6, 6))
wedges, texts, autotexts = ax.pie(
    category_tokens.values(),
    labels=category_tokens.keys(),
    autopct="%1.1f%%",
    colors=colors,
    startangle=140,
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
for t in autotexts:
    t.set_fontsize(9)
ax.set_title("Token Distribution by Category", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ============================================================
# CELL 4 — Chunk Size Distribution (after text splitting)
# ============================================================
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(chunk_lengths, bins=30, color="#4C72B0", edgecolor="white", linewidth=0.8)
ax.axvline(
    np.mean(chunk_lengths),
    color="red",
    linestyle="--",
    label=f"Mean: {np.mean(chunk_lengths):.0f}",
)
ax.axvline(
    np.median(chunk_lengths),
    color="orange",
    linestyle="--",
    label=f"Median: {np.median(chunk_lengths):.0f}",
)
ax.set_title("Chunk Size Distribution (characters)", fontsize=14, fontweight="bold")
ax.set_xlabel("Chunk Size (chars)")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.show()

# ============================================================
# CELL 5 — Chunks per doc_category
# ============================================================
chunk_by_category = Counter(
    chunk.metadata.get("doc_category", "unknown") for chunk in chunks
)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.barh(
    list(chunk_by_category.keys()),
    list(chunk_by_category.values()),
    color=colors[: len(chunk_by_category)],
    edgecolor="white",
    linewidth=1.5,
)
ax.bar_label(bars, padding=4, fontsize=11, fontweight="bold")
ax.set_title("Number of Chunks per Document Category", fontsize=14, fontweight="bold")
ax.set_xlabel("Number of Chunks")
plt.tight_layout()
plt.show()

# ============================================================
# CELL 6 — 2D Embedding Visualisation with UMAP/TSNE
# ============================================================
# Requires: pip install umap-learn
import umap
from sklearn.preprocessing import LabelEncoder

texts = [c.page_content for c in chunks]
labels = [c.metadata.get("doc_category", "unknown") for c in chunks]

embeddings_raw = embedding_model.embed_documents(texts)
embeddings_np = np.array(embeddings_raw)

reducer = umap.UMAP(n_components=2, random_state=42)
reduced = reducer.fit_transform(embeddings_np)

unique_labels = list(set(labels))
color_map = {
    lbl: plt.cm.tab10(i / len(unique_labels)) for i, lbl in enumerate(unique_labels)
}

fig, ax = plt.subplots(figsize=(10, 7))
for lbl in unique_labels:
    idx = [i for i, l in enumerate(labels) if l == lbl]
    ax.scatter(
        reduced[idx, 0],
        reduced[idx, 1],
        label=lbl,
        color=color_map[lbl],
        s=40,
        alpha=0.75,
        edgecolors="none",
    )

ax.set_title(
    "UMAP Projection of Document Chunk Embeddings", fontsize=14, fontweight="bold"
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(title="Doc Type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()